# 03 · Detecting benchmark contamination with n-grams

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/evals/03_contamination.ipynb) &nbsp; 📖 Concept deck: [Contamination](https://nilesh-salpe.github.io/visual-ai/evals/05-contamination/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** a GPT-3-style 13-gram overlap detector, a contamination report that splits the benchmark into clean and dirty subsets, and a simulation of how leaked test items inflate scores.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def sigmoid(z):
    return 1 / (1 + np.exp(-z))

rng = np.random.default_rng(0)
WORDS = [f"w{i}" for i in range(5000)]
def doc(n):
    return list(rng.choice(WORDS, n))
train_docs = [doc(int(rng.integers(200, 800))) for _ in range(300)]
test_items = [doc(int(rng.integers(20, 60))) for _ in range(200)]
LEAKED = set(rng.choice(200, 30, replace=False).tolist())
for i in LEAKED:                       # copy a span of each leaked test item into a random training doc
    item = test_items[i]; span = item[:max(15, len(item) // 2)]
    d = train_docs[int(rng.integers(300))]; pos = int(rng.integers(len(d))); d[pos:pos] = span
print(f"{len(train_docs)} training docs, {len(test_items)} test items, {len(LEAKED)} secretly leaked")

## Task 1: Collect n-grams

Return the **set** of all contiguous n-grams (as tuples) in a token list. If the list is shorter than n, return an empty set.

In [ ]:
def ngram_set(tokens, n):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_ngram_set(f):
    assert f(['a', 'b', 'c', 'd'], 2) == {('a', 'b'), ('b', 'c'), ('c', 'd')}
    assert f(['a', 'b'], 3) == set()
    assert len(f(['x'] * 10, 3)) == 1

check("ngram_set", _test_ngram_set, ngram_set)

## Task 2: Flag test items that overlap the training data

GPT-3 flagged test examples "that have a 13-gram overlap with anything in the pretraining set (or that overlap with the whole example when it is shorter than 13-grams)". The goal was to "very conservatively flag anything that could potentially be contamination".
Build one n-gram set over all training docs, then return a boolean array: is any n-gram of each test item in it? For items shorter than n, check whether the whole item appears as a contiguous span of some training doc (you may use a set of all len(item)-grams).

In [ ]:
def flag_contaminated(test_items, train_docs, n=13):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_flag_contaminated(f):
    flags = f(test_items, train_docs)
    assert flags.dtype == bool and len(flags) == 200
    found = set(np.nonzero(flags)[0].tolist())
    assert LEAKED <= found, f"missed leaked items: {sorted(LEAKED - found)}"
    assert len(found - LEAKED) == 0, f"false positives: {sorted(found - LEAKED)}"
    short = [['w1', 'w2', 'w3']]
    assert f(short, [['w0', 'w1', 'w2', 'w3', 'w4']])[0] and not f(short, [['w1', 'w3', 'w2']])[0], "short items: match the whole item as a span"

check("flag_contaminated", _test_flag_contaminated, flag_contaminated)

## Task 3: Clean vs dirty scores

GPT-3's approach: score the model on the full benchmark, on the **clean** subset (not flagged) and on the **dirty** subset (flagged). If clean ≈ full, contamination isn't driving the score.
Given per-item 0/1 `scores` and boolean `flags`, return a dict with `full`, `clean`, `dirty` accuracies and `frac_dirty`.

In [ ]:
def contamination_report(scores, flags):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_contamination_report(f):
    s = np.array([1, 1, 0, 1, 0, 1.0]); fl = np.array([True, True, False, False, False, False])
    r = f(s, fl)
    assert np.isclose(r['full'], 4 / 6) and np.isclose(r['dirty'], 1.0) and np.isclose(r['clean'], 0.5) and np.isclose(r['frac_dirty'], 2 / 6)

check("contamination_report", _test_contamination_report, contamination_report)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Collect n-grams
def ngram_set_solution(tokens, n):
    return {tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)}

check("ngram_set (reference)", _test_ngram_set, ngram_set_solution)

In [ ]:
# @title Solution for Task 2: Flag test items that overlap the training data
def flag_contaminated_solution(test_items, train_docs, n=13):
    grams = set().union(*(ngram_set_solution(d, n) for d in train_docs))
    out = []
    for item in test_items:
        if len(item) < n:
            m = len(item)
            out.append(any(tuple(item) in ngram_set_solution(d, m) for d in train_docs))
        else:
            out.append(not grams.isdisjoint(ngram_set_solution(item, n)))
    return np.array(out, dtype=bool)

check("flag_contaminated (reference)", _test_flag_contaminated, flag_contaminated_solution)

In [ ]:
# @title Solution for Task 3: Clean vs dirty scores
def contamination_report_solution(scores, flags):
    s = np.asarray(scores, float); fl = np.asarray(flags, bool)
    return dict(full=s.mean(), clean=s[~fl].mean(), dirty=s[fl].mean() if fl.any() else float('nan'), frac_dirty=fl.mean())

check("contamination_report (reference)", _test_contamination_report, contamination_report_solution)

---
## Experiments
**How much does leakage inflate a score?** Simulate a model with true skill 55% that answers leaked items correctly 95% of the time (memorized). Sweep the leaked fraction and compare the reported score with the clean-subset score. Then run the detector at different n: smaller n catches paraphrase-like partial overlaps but also flags coincidental matches on real text, which is why GPT-3 chose a conservative 13. (Our random-word toy has no coincidental matches; real text does.)

In [ ]:
r = np.random.default_rng(1)
for frac in [0.0, 0.05, 0.1, 0.2, 0.4]:
    fl = r.random(2000) < frac
    s = np.where(fl, r.random(2000) < 0.95, r.random(2000) < 0.55).astype(float)
    rep = contamination_report_solution(s, fl)
    print(f"leaked {frac:4.0%}: reported {rep['full']:.3f}   clean subset {rep['clean']:.3f}   inflation {100 * (rep['full'] - rep['clean']):+.1f} pts")
for n in [8, 13, 20]:
    print(f"n={n:2d}: flagged {flag_contaminated_solution(test_items, train_docs, n).sum()} of 200 (30 truly leaked)")

---
## Takeaways
* Contamination means test items (or close variants) appeared in training data, so the score partly measures memorization.
* n-gram overlap (GPT-3 used 13-grams) is a simple, conservative detector; report clean-subset scores next to full ones.
* Detection misses paraphrases and translations. Fresh, held-out test sets (GSM1k found drops of up to 8% for some model families) and canary strings are complementary defences.